# 04 — Feature-time ledger

**Objective.** Construct strict event-timed and snapshot-extended feature matrices, then enforce the availability ledger and prohibited-field guards.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Headline conclusions must survive the strict feature-set analysis or be labelled dependent on snapshot-static metadata.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("04", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
from cruxvc.features import build_features
from cruxvc.io import read_json, read_table, write_json, write_table

inputs = [
    P.processed / "cohort_labels.parquet",
    P.interim / "companies_canonical.parquet",
    P.interim / "funding_events.parquet",
    P.interim / "investment_edges.parquet",
]
CTX.recorder.inputs.extend(inputs)
cohort, companies, funding, investments = [read_table(path) for path in inputs]

In [ ]:
result = build_features(cohort, companies, funding, investments)
strict_path = write_table(result.strict, P.processed / "features_strict.parquet")
extended_path = write_table(result.extended, P.processed / "features_extended.parquet")
ledger_path = write_table(result.ledger, P.protocol / "feature_time_ledger.csv")
audit_path = write_json(result.audit, P.audits / "04_feature_audit.json")

In [ ]:
source_manifest = read_json(P.protocol / "source_manifest.json")
expected = {
    "landmark_amount_missing": 659,
    "founding_date_missing": 607,
    "founding_date_invalid": 418,
    "landmark_investor_edge_missing": 1074,
}
if source_manifest["all_expected_hashes_match"] and CFG["execution"]["strict_expected_counts_when_hashes_match"]:
    differences = {key: (expected[key], result.audit[key]) for key in expected if result.audit[key] != expected[key]}
    if differences:
        raise RuntimeError(f"Feature audit did not reproduce plan-derived counts: {differences}")

In [ ]:
CTX.recorder.complete([strict_path, extended_path, ledger_path, audit_path])
print(result.audit)
print(result.ledger.groupby(["availability_class", "approved"]).size())